# Buat `submission.csv` dari hasil OCR yang tersimpan

Notebook mandiri: hanya membaca `test.csv` dan cache OCR (`ocr_yolo.json` atau `ocr_full.json`), tanpa YOLO maupun EasyOCR.
Jalankan sel dari atas ke bawah. Format keluaran: kolom `id,text`; teks huruf besar, hanya `A-Z 0-9 / -`, tanpa spasi atau baris baru.

In [1]:
import csv
import json
import re
import pandas as pd

TEST_CSV  = 'test.csv'        # daftar id test
OCR_CACHE = 'ocr_yolo.json'   # atau 'ocr_full.json' untuk hasil mode full
OUT_CSV   = 'submission.csv'
MIN_CONF  = 0.15              # buang bacaan EasyOCR ber-confidence di bawah ini (hanya untuk crop)

In [2]:
# Fungsi pembersih & penyusun teks (definisi saja)
_DASH = str.maketrans({'–': '-', '—': '-', '‒': '-', '−': '-', '_': '-', '\\': '/'})
_VALID = re.compile(r'[^A-Z0-9/-]')
_FORMAT = re.compile(r'[A-Z0-9/-]+')


def clean_text(s):
    """Huruf besar; hanya A-Z, 0-9, '/' dan '-'; semua spasi/baris baru hilang."""
    return _VALID.sub('', s.translate(_DASH).upper())


def to_items(dets):
    items = []
    for d in dets:
        xs = [p[0] for p in d['box']]
        ys = [p[1] for p in d['box']]
        x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
        w, h = x2 - x1, y2 - y1
        txt = clean_text(d['text'])
        items.append(dict(x1=x1, x2=x2, w=w, h=h, cy=(y1 + y2) / 2, text=txt, conf=d['conf'],
                          vertical=(h > 1.5 * w and len(txt) >= 2)))
    return [it for it in items if it['text']]


def merge_lines(items, y_tol=0.6, gap_tol=2.5):
    """Gabungkan potongan teks yang sebaris (EasyOCR sering memecah satu ID menjadi beberapa bagian)."""
    mean = lambda v: sum(v) / len(v)
    lines = []
    for it in sorted([i for i in items if not i['vertical']], key=lambda i: i['cy']):
        for ln in lines:
            cy = mean([m['cy'] for m in ln])
            hh = max(it['h'], mean([m['h'] for m in ln]))
            gap = min(max(0, it['x1'] - m['x2'], m['x1'] - it['x2']) for m in ln)
            if abs(it['cy'] - cy) < y_tol * hh and gap < gap_tol * hh:
                ln.append(it)
                break
        else:
            lines.append([it])
    out = []
    for ln in lines:
        ln = sorted(ln, key=lambda m: m['x1'])
        wts = [len(m['text']) for m in ln]
        out.append(dict(text=''.join(m['text'] for m in ln), cy=mean([m['cy'] for m in ln]),
                        conf=sum(m['conf'] * w for m, w in zip(ln, wts)) / sum(wts)))
    out += [dict(text=i['text'], cy=i['cy'], conf=i['conf']) for i in items if i['vertical']]
    return out


def crop_text(dets, min_conf):
    """Isi crop `entete`: semua baris dibaca atas -> bawah lalu digabung tanpa spasi."""
    lines = merge_lines([it for it in to_items(dets) if it['conf'] >= min_conf])
    lines.sort(key=lambda l: l['cy'])
    return ''.join(l['text'] for l in lines).strip('/-')


def score_line(ln):
    t = ln['text']
    if len(t.strip('/-')) < 3:
        return 0.0
    s = ln['conf'] * min(len(t), 20)
    if any(c.isdigit() for c in t) and any(c in '/-' for c in t):   # pola khas ID: angka + pemisah
        s *= 1.5
    return s


def best_line_text(dets):
    """Cadangan / mode full: pilih SATU baris terbaik dari seluruh gambar."""
    lines = merge_lines(to_items(dets))
    best = max(lines, key=score_line, default=None)
    if best is not None and score_line(best) > 0:
        return best['text'].strip('/-') or '0'
    return '0'


def final_text(entry, min_conf):
    if entry.get('src') == 'crop':
        return crop_text(entry['dets'], min_conf) or '0'
    return best_line_text(entry.get('dets', []))

In [3]:
df_test = pd.read_csv(TEST_CSV, dtype=str)
id_column = df_test.columns[0]
ids = df_test[id_column].tolist()

with open(OCR_CACHE, encoding='utf-8') as f:
    cache = json.load(f)

n_missing = sum(i not in cache for i in ids)
if n_missing:
    print(f"PERINGATAN: {n_missing} gambar belum di-OCR (diisi '0'). Selesaikan OCR dulu, lalu jalankan ulang sel ini.")

preds = [final_text(cache[i], MIN_CONF) if i in cache else '0' for i in ids]
submission_df = pd.DataFrame({id_column: ids, 'text': preds})

# Validasi format sesuai aturan lomba
assert submission_df['text'].str.fullmatch(r'[A-Z0-9/-]+').all(), "Ada teks yang melanggar format"
assert submission_df[id_column].is_unique and len(submission_df) == len(df_test)

submission_df.to_csv(OUT_CSV, index=False)
print(f"{OUT_CSV} tersimpan: {submission_df.shape}")
print("Berisi '0' (tidak terbaca / belum di-OCR):", int((submission_df['text'] == '0').sum()))
print("Panjang teks -> min/median/maks:", submission_df['text'].str.len().agg(['min', 'median', 'max']).tolist())
submission_df.head(10)

submission.csv tersimpan: (674, 2)
Berisi '0' (tidak terbaca / belum di-OCR): 0
Panjang teks -> min/median/maks: [2.0, 20.0, 103.0]


,id,text
0,0016f1ad9e7f16aa6dd2b8c9c13cb922.jpg,11I1
1,005e28f75a93bc4de0e9e11373bdca47.jpg,EXPAFEV23FABAFEV20LOT395
2,007f7b00eb0a66cabcd72007358e3660.jpg,FABLOT17LM8
3,009a43c2870949af417f769b90ee20cf.jpg,1EXP0872021LOTS2F834731FAB0922018
4,014d3d8a2ef0d4deb88a83357781745c.jpg,1EURAPHAULOTH5154
5,0174f52895414f9f2b5842547803f533.jpg,11DE3250SHPPX7910DAAMP831305048004003SODIUM8BA...
6,02a45854f6bdaff85587d8017a344ff9.jpg,46128171
7,02d5c041d5aebd133a34b7cb5e679557.jpg,1EXP-0612021FAB07/2018LOTX34549
8,02ff1c092c2f1d483916e898ef740392.jpg,10PERH04-2024378111
9,0396159a179dc766efd00a86fa040f02.jpg,1PPAA12406-0QOSHP-12406DA1L12008DOLIPRANE18ANO...
